In [1]:
%load_ext autoreload
%autoreload 2

In [13]:
from concept_abstraction.training import train_model, train_ppo_model
from concept_abstraction.selection import greedy_selection, random_selection
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
from concept_abstraction.environments import *
from sklearn.metrics import accuracy_score
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
import gymnasium as gym
from collections import Counter
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import SubprocVecEnv
import ocatari
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import numpy as np
import gymnasium as gym
import torch


In [3]:
is_jupyter = 'ipykernel' in sys.modules

In [4]:
if is_jupyter: 
    seed        = 42
    environment_string = "cartpole"
    concept_retrieval = "raw"
    show_baseline = True 
    human_accuracy_by_concept = None 
    target_abstraction = 0.05
    out_folder = "cartpole"
    num_concepts_selected = 4
    cbm_accuracy_by_concept = None 
    human_reliance_by_concept = None 
    reward_error = 0.1
    transition_error = 0
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
    parser.add_argument('--environment_nodes', help='Size of the environment; number of nodes', type=int, default=4)
    parser.add_argument('--show-baseline', action='store_true', help='Whether to show the baseline')
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--human_accuracy_by_concept', nargs='*', type=float, default=None)
    parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
    parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
    parser.add_argument('--human_reliance_by_concept', help="How much does AI rely on human intervention?",  nargs='*', type=float, default=None)
    parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
    parser.add_argument('--transition_error', help="How much to perturb the transition by?", type=float, default=0)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    environment_string = args.environment_string
    environment_nodes = args.environment_nodes 
    show_baseline = args.show_baseline
    num_concepts_selected = args.num_concepts_selected
    human_accuracy_by_concept = args.human_accuracy_by_concept
    human_reliance_by_concept = args.human_reliance_by_concept
    target_abstraction = args.target_abstraction
    cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
    reward_error = args.reward_error
    transition_error = args.transition_error
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [5]:
results = {}
results['parameters'] = {'seed'      : seed,
        'environment_string'    : environment_string, 
        'show_baseline': show_baseline,
        'num_concepts_selected': num_concepts_selected,
        'human_accuracy_by_concept': human_accuracy_by_concept, 
        'human_reliance_by_concept': human_reliance_by_concept, 
        'target_abstraction': target_abstraction,
        'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
        'reward_error': reward_error, 
        'transition_error': transition_error,
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'cartpole', 'show_baseline': True, 'num_concepts_selected': 4, 'human_accuracy_by_concept': None, 'human_reliance_by_concept': None, 'target_abstraction': 0.05, 'cbm_accuracy_by_concept': None, 'reward_error': 0.1, 'transition_error': 0}


In [6]:
np.random.seed(seed)
random.seed(seed)

In [7]:
def make_env_fn(concept_list,accuracies=None,binary=False,aggregated=False,llm=False,reward_error=0):
    env = gym.make("CartPole-v1")

    if binary:
        env = DiscretizeObservationWrapper(env, bins_per_feature=4)
        env = BinaryObservationSubsetWrapper(env, concept_list,accuracies)
        env.concepts = list(range(16))
    elif aggregated:
        env = get_binary_subset_env(golden_model, env, concept_list,accuracies=accuracies)
    elif llm:
        env = CustomBinaryFeatureWrapper(env)
        env = BinaryObservationSubsetWrapper(env, concept_list,accuracies=accuracies)
    else:
        env = ObservationSubsetWrapper(env, indices=concept_list)

    if reward_error > 0:
        env = RewardPerturbationWrapper(env,reward_error)

    return env

In [8]:
total_timesteps = 20000

### Testing out Breakout

#### CNN

In [10]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecFrameStack, VecTransposeImage
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.evaluation import evaluate_policy

# 1. Create Breakout env with proper preprocessing
env_id = "BreakoutNoFrameskip-v4"
vec_env = make_atari_env(env_id, n_envs=8, seed=0)
vec_env = VecFrameStack(vec_env, n_stack=4)
vec_env = VecTransposeImage(vec_env)  # channel-last for PyTorch

# 2. Init PPO
model = PPO("CnnPolicy", vec_env, verbose=0)

# 3. Train + evaluate loop
timesteps_per_iter = 100_000
for i in range(5):  # total = 500k
    print(f"\n=== Iteration {i+1} ===")
    model.learn(total_timesteps=timesteps_per_iter, reset_num_timesteps=False)
    
    mean_reward, std_reward = evaluate_policy(model, vec_env, n_eval_episodes=10)
    print(f"Mean reward: {mean_reward:.2f} ± {std_reward:.2f}")


A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]



=== Iteration 1 ===


KeyboardInterrupt: 

#### Concepts

In [101]:

# RAM normalization wrapper
class NormalizeRAM(gym.ObservationWrapper):
    def observation(self, obs):
        obs = np.array(obs, dtype=np.float32) / 255.0
        return obs

# Make a single Pong RAM env with easy opponent
def make_ocenv(seed=0):
    env = ocatari.OCAtari(
        "PongNoFrameskip-v4",
        mode="ram",
        render_mode=None,
        frameskip=1,  # Keep frameskip=1 so consecutive frames actually differ
        difficulty=0  # easiest opponent
    )
    env = NormalizeRAM(env)
    env = Monitor(env)  # Add Monitor wrapper
    env.reset(seed=seed)
    return env

n_envs = 8  # Use up to 16 or CPU count

# Vectorized environment WITH frame stacking (you were right!)
vec_env = SubprocVecEnv([
    lambda seed=i: make_ocenv(seed=seed) for i in range(n_envs)
], start_method='spawn')  # 'spawn' is more stable across platforms

vec_env = VecFrameStack(vec_env, n_stack=4)

# PPO with larger network to handle 4x RAM input (512 values instead of 128)
model = PPO(
    "MlpPolicy",
    vec_env,
    policy_kwargs=dict(
        net_arch=[512, 256, 128],  # Larger first layer for 4x128 RAM input
        activation_fn=torch.nn.ReLU
    ),
    verbose=0,
    learning_rate=0.0003,  # Lower learning rate
    n_steps=2048,          # More steps per update
    batch_size=64,         # Reasonable batch size
    n_epochs=10,           # More epochs per update
    gamma=0.99,           # Standard discount factor
    gae_lambda=0.95,      # Standard GAE lambda
    clip_range=0.2,       # Standard clip range
    ent_coef=0.01         # Small entropy coefficient
)

eval_env = VecFrameStack(
    SubprocVecEnv([lambda: make_ocenv(seed=42)], start_method='spawn'), 
    n_stack=4
)

# OPTIMIZATION 5: Larger training chunks for efficiency
timesteps_per_iter = 100000  # Doubled chunk size
total_iters = 15  # 1.5M timesteps total

print("Training with VecFrameStack for temporal information (velocity, trajectories)")
print(f"Input shape: 4 stacked RAM states = 512 values")
print(f"This captures ball/paddle positions across 4 consecutive frames")

for i in range(total_iters):
    print(f"\n=== Training iteration {i+1}/{total_iters} ===")
    model.learn(total_timesteps=timesteps_per_iter, reset_num_timesteps=False)
    
    # Evaluation every few iterations
    if (i + 1) % 4 == 0:  # Evaluate every 4 iterations
        mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=5)
        print(f"Mean reward: {mean_reward:.2f} ± {std_reward:.2f}")

# Final evaluation
print("\n=== Final Evaluation ===")
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10)
print(f"Final mean reward: {mean_reward:.2f} ± {std_reward:.2f}")


A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]


Training with VecFrameStack for temporal information (velocity, trajectories)
Input shape: 4 stacked RAM states = 512 values
This captures ball/paddle positions across 4 consecutive frames

=== Training iteration 1/15 ===

=== Training iteration 2/15 ===

=== Training iteration 3/15 ===

=== Training iteration 4/15 ===
Mean reward: -21.00 ± 0.00

=== Training iteration 5/15 ===


KeyboardInterrupt: 

In [14]:
import ocatari
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import numpy as np
import gymnasium as gym
import torch

# RAM normalization wrapper
class NormalizeRAM(gym.ObservationWrapper):
    def observation(self, obs):
        obs = np.array(obs, dtype=np.float32) / 255.0
        return obs

# Make a single Pong RAM env with easy opponent
def make_ocenv(seed=0):
    env = ocatari.OCAtari(
        "PongNoFrameskip-v4",
        mode="ram",
        render_mode=None,
        frameskip=1,  # Keep frameskip=1 so consecutive frames actually differ
        difficulty=0  # easiest opponent
    )
    env = NormalizeRAM(env)
    env = Monitor(env)  # Add Monitor wrapper
    env.reset(seed=seed)
    return env

n_envs = 8  # Use up to 16 or CPU count

# Vectorized environment WITH frame stacking (you were right!)
vec_env = SubprocVecEnv([
    lambda seed=i: make_ocenv(seed=seed) for i in range(n_envs)
], start_method='spawn')  # 'spawn' is more stable across platforms

vec_env = VecFrameStack(vec_env, n_stack=4)



A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]


In [40]:
env = ocatari.OCAtari(
    "PongNoFrameskip-v4",
    mode="ram",
    render_mode=None,
    frameskip=1,  # Keep frameskip=1 so consecutive frames actually differ
    difficulty=0  # easiest opponent
)
print(env.objects)
env.reset()
for i in range(100):
    env.step(1)
    print(env.objects,env.step(1)[0][0])

[Player at (0, 0), (4, 15), Ball at (0, 0), (2, 4), Enemy at (0, 0), (4, 15)]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96   0   0   0   0]
[Player at (140, 96), (4, 15), NaO, NaO] [140  96  

In [49]:
class RAMVelocityWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        # We'll return a 7-dimensional float32 vector
        self.observation_space = gym.spaces.Box(low=-255, high=255, shape=(7,), dtype=np.float32)
        self.prev_ram = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.prev_ram = obs
        return self._process(obs), info

    def observation(self, obs):
        processed = self._process(obs)
        self.prev_ram = obs
        return processed

    def _process(self, obs):
        obs = np.array(obs, dtype=np.float32)
        # extract the 6 values (assuming your obs shape is (4,6))
        # last row is the newest frame
        current = obs[-1]
        previous = obs[-2] if obs.shape[0] >= 2 else current
        # construct your custom vector: [pos1, pos2, pos3, vel1, vel2, pos4, vel3]
        vec = np.array([
            current[1],       # 96
            current[2],       # 102
            current[3],       # 53
            current[2]-previous[2],  # 102-101
            current[3]-previous[3],  # 53-51
            current[5],       # 51
            current[5]-previous[5]   # 51-49
        ], dtype=np.float32)
        return vec

(array([96.,  0.,  0.,  0.,  0.,  0.,  0.], dtype=float32), 0.0, False, False, {'lives': 0, 'episode_frame_number': 1, 'frame_number': 1})
(array([98.,  0.,  0.,  0.,  0.,  0.,  0.], dtype=float32), 0.0, False, False, {'lives': 0, 'episode_frame_number': 2, 'frame_number': 2})
(array([98.,  0.,  0.,  0.,  0.,  0.,  0.], dtype=float32), 0.0, False, False, {'lives': 0, 'episode_frame_number': 3, 'frame_number': 3})
(array([105.,   0.,   0.,   0.,   0.,   0.,   0.], dtype=float32), 0.0, False, False, {'lives': 0, 'episode_frame_number': 4, 'frame_number': 4})
(array([105.,   0.,   0.,   0.,   0.,   0.,   0.], dtype=float32), 0.0, False, False, {'lives': 0, 'episode_frame_number': 5, 'frame_number': 5})
(array([115.,   0.,   0.,   0.,   0.,   0.,   0.], dtype=float32), 0.0, False, False, {'lives': 0, 'episode_frame_number': 6, 'frame_number': 6})
(array([115.,   0.,   0.,   0.,   0.,   0.,   0.], dtype=float32), 0.0, False, False, {'lives': 0, 'episode_frame_number': 7, 'frame_number': 7})

In [51]:
import ocatari
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import numpy as np
import gymnasium as gym
import torch

# RAM normalization wrapper
class NormalizeRAM(gym.ObservationWrapper):
    def observation(self, obs):
        obs = np.array(obs, dtype=np.float32) / 255.0
        return obs

# Make a single Pong RAM env with easy opponent
def make_ocenv(seed=0):
    env = ocatari.OCAtari("PongNoFrameskip-v4", mode="ram", frameskip=1, difficulty=0)
    env = RAMVelocityWrapper(env)
    env = NormalizeRAM(env)
    env = Monitor(env)  # Add Monitor wrapper
    env.reset(seed=seed)
    return env

n_envs = 8  # Use up to 16 or CPU count

# Vectorized environment WITH frame stacking (you were right!)
vec_env = SubprocVecEnv([
    lambda seed=i: make_ocenv(seed=seed) for i in range(n_envs)
], start_method='spawn')  # 'spawn' is more stable across platforms

vec_env = VecFrameStack(vec_env, n_stack=4)


# PPO with larger network to handle 4x RAM input (512 values instead of 128)
model = PPO(
    "MlpPolicy",
    vec_env,
    policy_kwargs=dict(
        net_arch=[512, 256, 128],  # Larger first layer for 4x128 RAM input
        activation_fn=torch.nn.ReLU
    ),
    verbose=0,
    learning_rate=0.0003,  # Lower learning rate
    n_steps=2048,          # More steps per update
    batch_size=64,         # Reasonable batch size
    n_epochs=10,           # More epochs per update
    gamma=0.99,           # Standard discount factor
    gae_lambda=0.95,      # Standard GAE lambda
    clip_range=0.2,       # Standard clip range
    ent_coef=0.01         # Small entropy coefficient
)

eval_env = VecFrameStack(
    SubprocVecEnv([lambda: make_ocenv(seed=42)], start_method='spawn'), 
    n_stack=4
)

# OPTIMIZATION 5: Larger training chunks for efficiency
timesteps_per_iter = 100000  # Doubled chunk size
total_iters = 15  # 1.5M timesteps total

print("Training with VecFrameStack for temporal information (velocity, trajectories)")
print(f"Input shape: 4 stacked RAM states = 512 values")
print(f"This captures ball/paddle positions across 4 consecutive frames")

for i in range(total_iters):
    print(f"\n=== Training iteration {i+1}/{total_iters} ===")
    model.learn(total_timesteps=timesteps_per_iter, reset_num_timesteps=False)
    
    # Evaluation every few iterations
    if (i + 1) % 4 == 0:  # Evaluate every 4 iterations
        mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=5)
        print(f"Mean reward: {mean_reward:.2f} ± {std_reward:.2f}")

# Final evaluation
print("\n=== Final Evaluation ===")
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10)
print(f"Final mean reward: {mean_reward:.2f} ± {std_reward:.2f}")


A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+01d3e3a)
[Powered by Stella]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/124

Training with VecFrameStack for temporal information (velocity, trajectories)
Input shape: 4 stacked RAM states = 512 values
This captures ball/paddle positions across 4 consecutive frames

=== Training iteration 1/15 ===

=== Training iteration 2/15 ===

=== Training iteration 3/15 ===

=== Training iteration 4/15 ===
Mean reward: -20.00 ± 0.00

=== Training iteration 5/15 ===

=== Training iteration 6/15 ===

=== Training iteration 7/15 ===

=== Training iteration 8/15 ===
Mean reward: 21.00 ± 0.00

=== Training iteration 9/15 ===

=== Training iteration 10/15 ===

=== Training iteration 11/15 ===

=== Training iteration 12/15 ===
Mean reward: 17.00 ± 0.00

=== Training iteration 13/15 ===


KeyboardInterrupt: 